In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# 数据类型定义（与 utils.py 保持一致）
DTYPE_PV = "f8"
DTYPE_ID = "i8"
DTYPE_ENUM = "S8"
DTYPE_TIME = "S16"
DTYPE_TIMESTAMP = "f8"

DTYPE_MD = np.dtype([
    ("mdtype", DTYPE_ENUM),          # insert/cancel/trade
    ("symbol", DTYPE_ENUM),          # 标的名称
    ("exchange_timestamp", DTYPE_TIMESTAMP),  # 交易所时间戳
    ("exchange_time", DTYPE_TIME),   # 交易所时间
    ("oid", DTYPE_ID),               # 订单ID
    ("buy_oid", DTYPE_ID),           # 买方订单ID
    ("sell_oid", DTYPE_ID),          # 卖方订单ID
    ("seq", DTYPE_ID),               # 事件发生的顺序
    ("price", DTYPE_PV),             # 价格
    ("volume", DTYPE_PV),            # 数量
    ("side", DTYPE_ENUM),            # 买卖方向 B=买, S=卖
    ("type", DTYPE_ENUM),            # 订单类型 50=限价单, 49/85=市价单
], align=True)

# 读取行情数据函数
def read_md(date, symbol):
    mdpath = Path("marketdata") / date / f"{symbol}.bin"
    return pd.DataFrame(np.fromfile(mdpath, dtype=DTYPE_MD))

# 读取ground truth数据函数
def read_output(date, name):
    outputpath = Path("ground_truth") / name / date
    symbols = sorted([p.stem for p in outputpath.glob("*.bin")])
    outputs = {symbol: np.fromfile(outputpath / f"{symbol}.bin", dtype=DTYPE_PV) for symbol in symbols}
    index = [
        "09:31:00.000", "09:40:00.000", "09:50:00.000", "10:00:00.000",
        "10:10:00.000", "10:20:00.000", "10:30:00.000", "10:40:00.000",
        "10:50:00.000", "11:00:00.000", "11:10:00.000", "11:20:00.000",
        "11:30:05.000", "13:10:00.000", "13:20:00.000", "13:30:00.000",
        "13:40:00.000", "13:50:00.000", "14:00:00.000", "14:10:00.000",
        "14:20:00.000", "14:30:00.000", "14:40:00.000", "14:50:00.000",
        "14:56:00.000", "14:57:05.000", "15:05:00.000",
    ]
    return pd.DataFrame(outputs, index=[f"{date} {i}" for i in index])

# 读取数据
date = "2025-01-02"
symbol = "000690"
output_name = "output_1"

# 读取行情数据
md = read_md(date, symbol)

# 读取ground truth数据
gt = read_output(date, output_name)

print("=" * 60)
print("行情数据特征描述:")
print("=" * 60)
print(f"数据类型: {md.dtypes.to_dict()}")
print(f"\n数据形状: {md.shape} (共 {md.shape[0]} 条记录, {md.shape[1]} 个特征)")
print(f"\n各字段说明:")
print("  mdtype          - 事件类型 (insert=挂单, cancel=撤单, trade=成交)")
print("  symbol          - 标的代码")
print("  exchange_timestamp - 交易所时间戳 (纳秒)")
print("  exchange_time   - 交易所时间 (HH:MM:SS.mmm)")
print("  oid             - 订单ID")
print("  buy_oid/sell_oid - 买方/卖方订单ID (trade事件)")
print("  seq             - 事件顺序")
print("  price           - 价格")
print("  volume          - 数量")
print("  side            - 方向 (B=买, S=卖)")
print("  type            - 订单类型 (50=限价单, 49/85=市价单)")

print("\n" + "=" * 60)
print("行情数据前五行:")
print("=" * 60)
print(md.head())

print("\n" + "=" * 60)
print("Ground Truth数据描述:")
print("=" * 60)
print(f"数据形状: {gt.shape} (共 {gt.shape[0]} 个时间点, {gt.shape[1]} 只股票)")
print(f"\n时间点: 共27个, 从 09:31:00.000 到 15:05:00.000")
print(f"股票数量: {gt.shape[1]} 只")
print(f"\n股票列表: {list(gt.columns)}")

print("\n" + "=" * 60)
print("Ground Truth数据前五行:")
print("=" * 60)
print(gt.head())

行情数据特征描述:
数据类型: {'mdtype': dtype('O'), 'symbol': dtype('O'), 'exchange_timestamp': dtype('float64'), 'exchange_time': dtype('O'), 'oid': dtype('int64'), 'buy_oid': dtype('int64'), 'sell_oid': dtype('int64'), 'seq': dtype('int64'), 'price': dtype('float64'), 'volume': dtype('float64'), 'side': dtype('O'), 'type': dtype('O')}

数据形状: (54645, 12) (共 54645 条记录, 12 个特征)

各字段说明:
  mdtype          - 事件类型 (insert=挂单, cancel=撤单, trade=成交)
  symbol          - 标的代码
  exchange_timestamp - 交易所时间戳 (纳秒)
  exchange_time   - 交易所时间 (HH:MM:SS.mmm)
  oid             - 订单ID
  buy_oid/sell_oid - 买方/卖方订单ID (trade事件)
  seq             - 事件顺序
  price           - 价格
  volume          - 数量
  side            - 方向 (B=买, S=卖)
  type            - 订单类型 (50=限价单, 49/85=市价单)

行情数据前五行:
      mdtype     symbol  exchange_timestamp    exchange_time   oid  buy_oid  \
0  b'insert'  b'000690'        1.735781e+09  b'09:15:00.010'   481        0   
1  b'insert'  b'000690'        1.735781e+09  b'09:15:00.050'  1781        0   
2  

In [ ]:
# 将所有ground_truth数据整理成一张大表
import numpy as np
import pandas as pd
from pathlib import Path

DTYPE_PV = "f8"

# 27个时间点
time_points = [
    "09:31:00.000", "09:40:00.000", "09:50:00.000", "10:00:00.000",
    "10:10:00.000", "10:20:00.000", "10:30:00.000", "10:40:00.000",
    "10:50:00.000", "11:00:00.000", "11:10:00.000", "11:20:00.000",
    "11:30:05.000", "13:10:00.000", "13:20:00.000", "13:30:00.000",
    "13:40:00.000", "13:50:00.000", "14:00:00.000", "14:10:00.000",
    "14:20:00.000", "14:30:00.000", "14:40:00.000", "14:50:00.000",
    "14:56:00.000", "14:57:05.000", "15:05:00.000",
]

# 所有output名称
output_names = [f"output_{i}" for i in range(1, 18)]

# 收集所有数据
all_data = []

for output_name in output_names:
    output_path = Path("ground_truth") / output_name
    if not output_path.exists():
        continue
    
    for date_dir in sorted(output_path.iterdir()):
        if not date_dir.is_dir():
            continue
        date = date_dir.name
        
        for bin_file in sorted(date_dir.glob("*.bin")):
            symbol = bin_file.stem
            values = np.fromfile(bin_file, dtype=DTYPE_PV)
            
            # 为每个时间点创建一行
            for idx, (time_point, value) in enumerate(zip(time_points, values)):
                all_data.append({
                    "date": date,
                    "symbol": symbol,
                    "time_point": time_point,
                    "time_idx": idx,
                    "output_name": output_name,
                    "value": value
                })

# 创建DataFrame
df_gt = pd.DataFrame(all_data)

print(f"数据形状: {df_gt.shape}")
print(f"总记录数: {len(df_gt)}")
print(f"日期数: {df_gt['date'].nunique()} - {sorted(df_gt['date'].unique())}")
print(f"股票数: {df_gt['symbol'].nunique()} - {sorted(df_gt['symbol'].unique())}")
print(f"时间点数: {df_gt['time_idx'].nunique()}")
print(f"Output数: {df_gt['output_name'].nunique()} - {sorted(df_gt['output_name'].unique())}")

print("\n" + "=" * 60)
print("前10行数据:")
print("=" * 60)
print(df_gt.head(10))

# 保存为CSV
csv_path = Path("ground_truth_summary.csv")
df_gt.to_csv(csv_path, index=False)
print(f"\n已保存到: {csv_path.absolute()}")

In [3]:
df = pd.read_csv("ground_truth_csv/output_1.csv")
display(df.head())

,datetime,000548,000677,000690,000848,000949,001389,002079,002177,002307,...,300816,300947,300967,301005,301035,301049,301157,301393,301552,301578
0,2025-01-02-09:31:00.000,NaN,NaN,708.0,NaN,NaN,446.0,NaN,NaN,NaN,...,246.0,NaN,NaN,NaN,NaN,NaN,360.0,NaN,NaN,NaN
1,2025-01-02-09:40:00.000,NaN,NaN,867.0,NaN,NaN,1253.0,NaN,NaN,NaN,...,503.0,NaN,NaN,NaN,NaN,NaN,811.0,NaN,NaN,NaN
2,2025-01-02-09:50:00.000,NaN,NaN,669.0,NaN,NaN,593.0,NaN,NaN,NaN,...,327.0,NaN,NaN,NaN,NaN,NaN,559.0,NaN,NaN,NaN
3,2025-01-02-10:00:00.000,NaN,NaN,611.0,NaN,NaN,643.0,NaN,NaN,NaN,...,251.0,NaN,NaN,NaN,NaN,NaN,380.0,NaN,NaN,NaN
4,2025-01-02-10:10:00.000,NaN,NaN,911.0,NaN,NaN,469.0,NaN,NaN,NaN,...,243.0,NaN,NaN,NaN,NaN,NaN,368.0,NaN,NaN,NaN
